# Budgerigar：顺序敏感内容检索头迁移训练

旧 token bank/CTC 已达到 CER 0.707，但平均池化检索停滞。本阶段迁移旧核心权重，只新建双向 GRU 音频序列头、文本序列头和投影层，以保留字符顺序。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位旧内容 checkpoint
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.arctic_slt.{FEATURE_FINGERPRINT}.pt'
SOURCE_CHECKPOINT=WORK_ROOT/'checkpoints'/f'content_memory_{FEATURE_FINGERPRINT}'/'best.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert STATS_PATH.is_file(),STATS_PATH
assert SOURCE_CHECKPOINT.is_file(),SOURCE_CHECKPOINT
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)
source=torch.load(SOURCE_CHECKPOINT,map_location='cpu',weights_only=True)
print('source architecture:',source['architecture'],'source best row:',min(source['history'],key=lambda row:row['validation_cer']))

In [ ]:
#@title 3. 顺序敏感 InfoNCE 迁移训练
MAX_STEPS=400 #@param {type:'integer'}
BATCH_SIZE=4 #@param {type:'integer'}
if not torch.cuda.is_available(): raise RuntimeError('请选择 GPU runtime')
from budgerigar.content_data import CharacterVocabulary
from budgerigar.content_memory import ContentMemoryConfig
from budgerigar.train_content_memory import ContentTrainingConfig,train_content_memory
vocab=CharacterVocabulary()
RUN_DIR=WORK_ROOT/'checkpoints'/f'content_memory_sequence_{FEATURE_FINGERPRINT}'
model_config=ContentMemoryConfig(token_slots=128,update_stride=4,vocabulary_size=len(vocab.symbols),sequence_contrastive=True)
training=ContentTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,learning_rate=2e-4,contrastive_weight=1.0,initialization_checkpoint=str(SOURCE_CHECKPOINT))
report=train_content_memory(FEATURE_MANIFEST,stats,RUN_DIR,training,model_config)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 联合门槛检查
best=min(report['history'],key=lambda row:row['selection_score'])
print(json.dumps(best,ensure_ascii=False,indent=2))
sequence_pass=best['validation_cer']<0.7 and best['validation_retrieval_top1']>0.5
print('sequence_pass =',sequence_pass)

In [ ]:
#@title 5. 保存元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(RUN_DIR/'run_metadata.json',FEATURE_MANIFEST,{'architecture':'content_token_memory_sequence','initialization_checkpoint':str(SOURCE_CHECKPOINT),'best_validation_cer':report['best_validation_cer'],'best_selection_score':report['best_selection_score']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))